In [1]:
import rasterio
import xarray as xr
import glob
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from scipy.ndimage import median_filter
from rasterio.io import MemoryFile
from rasterio.features import shapes
from shapely.geometry import shape
#from skimage.morphology import (
#    binary_opening, binary_closing, binary_erosion, binary_dilation,
#    disk, remove_small_objects, remove_small_holes
#)
#from scipy.ndimage import binary_fill_holes

import skimage as ski
import numpy as np

In [2]:
years = [2023]

### 1) Define helper function for finding the data

In [3]:
def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [28]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/")
output_folder.mkdir(parents=True, exist_ok=True)

s2_base = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/S2_indices")
s1_base = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence")

### 2) Combine NDSI & Ratio files and create composite from east and west

In [34]:
def create_composite(ndsi_east_file, ndsi_west_file,
                     ratio_east_file, ratio_west_file,
                     output_file):

    # Merge east and west NDSI rasters
    with rasterio.open(ndsi_east_file) as east, \
         rasterio.open(ndsi_west_file) as west:

        ndsi, ndsi_transform = merge([east, west], method="max")
        meta = east.meta.copy()

    # Merge east and west ratio rasters
    with rasterio.open(ratio_east_file) as east, \
         rasterio.open(ratio_west_file) as west:

        ratio, ratio_transform = merge([east, west], method="max")

    # Combine NDSI and ratio using logical OR
    glacier_extent = (
        (ndsi[0] == 1) | (ratio[0] == 1)
    ).astype(np.uint8)

    # Save he final output
    meta.update({
        "height": glacier_extent.shape[0],
        "width": glacier_extent.shape[1],
        "transform": ndsi_transform,
        "count": 1,
        "dtype": "uint8",
        "nodata": 255,
    })

    with rasterio.open(output_file, "w", **meta) as dst:
        dst.write(glacier_extent, 1)

    print(f"Saved final glacier extent: {output_file}")

In [35]:
for year in years:

    print(f"\n===== {year} =====")

    input_folder_s2 = s2_base / str(year)

    # Find the four input rasters
    ndsi_east = find_files(
        input_folder_s2, "S2_mosaic_east_*_ndsi_*.tiff"
    )
    ndsi_west = find_files(
        input_folder_s2, "S2_mosaic_west_*_ndsi_*.tiff"
    )
    ratio_east = find_files(
        input_folder_s2, "S2_mosaic_east_*_ratio_*.tiff"
    )
    ratio_west = find_files(
        input_folder_s2, "S2_mosaic_west_*_ratio_*.tiff"
    )

    if not all([
        len(ndsi_east) == 1,
        len(ndsi_west) == 1,
        len(ratio_east) == 1,
        len(ratio_west) == 1,
    ]):
        print(f"Skipping {year}: expected exactly four input files.")
        continue

    # Final output only
    output_file = input_folder_s2 / f"glacier_composite_{year}.tiff"

    create_composite(
        ndsi_east[0],
        ndsi_west[0],
        ratio_east[0],
        ratio_west[0],
        output_file
    )


===== 2023 =====
Saved final glacier extent: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/S2_indices/2023/glacier_composite_2023.tiff


### 3) Mask ascending and descending mosaics with threshold 

In [5]:
thr = 0.4

In [36]:
def apply_threshold(input_filename, output_filename, thr=thr):
    with rasterio.open(input_filename) as src:
        data = src.read(1).astype(np.float32)
        meta = src.meta.copy()
        nodata = src.nodata

    # invalid = NaN or nodata value
    invalid = np.isnan(data)
    if nodata is not None and not np.isnan(nodata):
        invalid |= (data == nodata)

    # 1 = coherence below threshold, 0 = above, 255 = nodata
    glacier = np.where(data < thr, 1, 0).astype("uint8")
    glacier[invalid] = 255

    meta.update(count=1, dtype="uint8", nodata=255)

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(glacier, 1)

    print(f"Saved: {output_filename}")


for year in years:
    print(f"\n===== {year} =====")
    input_folder_s2 = s2_base / str(year)
    input_folder_s1 = s1_base / str(year)

    # find ascending / descending coherence files (note the f-strings)
    s1_coherence_files_asc = find_files(input_folder_s1, f"coh_mosaic_ascending_{year}.tiff")
    s1_coherence_files_desc = find_files(input_folder_s1, f"coh_mosaic_descending_{year}.tiff")
    print(f"Found {len(s1_coherence_files_asc)} S1 coherence files (ascending).")
    print(f"Found {len(s1_coherence_files_desc)} S1 coherence files (descending).")


    # apply threshold to S1 coherence files (pass file paths, not arrays)
    apply_threshold(
        s1_coherence_files_asc[0],
        input_folder_s1 / f"ascending_{year}_thresh_{thr}.tiff",
    )
    apply_threshold(
        s1_coherence_files_desc[0],
        input_folder_s1 / f"descending_{year}_thresh_{thr}.tiff",
    )


===== 2023 =====
Found 1 S1 coherence files (ascending).
Found 1 S1 coherence files (descending).


/tmp/ipykernel_1861977/571529502.py:3: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = src.read(1).astype(np.float32)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2023/ascending_2023_thresh_0.4.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/2023/descending_2023_thresh_0.4.tiff


### 4) Align all rasters spatially

In [37]:
def align_to_reference(src_array, src_transform, src_crs, ref_meta,
                       src_nodata=None, dst_nodata=None,
                       resampling=Resampling.nearest):
    fill = dst_nodata if dst_nodata is not None else 0
    aligned = np.full((ref_meta["height"], ref_meta["width"]), fill, dtype=src_array.dtype)
    reproject(
        source=src_array,
        destination=aligned,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=src_nodata,
        dst_transform=ref_meta["transform"],
        dst_crs=ref_meta["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    return aligned

In [38]:
for year in years:
    print(f"\n===== {year} =====")
    input_folder_s2 = s2_base / str(year)
    input_folder_s1 = s1_base / str(year)

    s2_file   = input_folder_s2 / f"glacier_composite_{year}.tiff"
    asc_file  = input_folder_s1 / f"ascending_{year}_thresh_{thr}.tiff"
    desc_file = input_folder_s1 / f"descending_{year}_thresh_{thr}.tiff"

    missing = [p.name for p in (s2_file, asc_file, desc_file) if not p.exists()]
    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    # load everything from disk
    with rasterio.open(asc_file) as src:
        s1_asc = src.read(1)
        s1_asc_meta = src.meta.copy()
    with rasterio.open(desc_file) as src:
        s1_desc = src.read(1)
        s1_desc_meta = src.meta.copy()
    with rasterio.open(s2_file) as src:
        s2 = src.read(1)
        s2_transform = src.transform
        s2_crs = src.crs

    ref_meta = s1_asc_meta  # reference grid

    s2_aligned = align_to_reference(
        s2, s2_transform, s2_crs, ref_meta, src_nodata=None, dst_nodata=0,
    )
    s1_asc_aligned = align_to_reference(
        s1_asc, s1_asc_meta["transform"], s1_asc_meta["crs"], ref_meta,
        src_nodata=255, dst_nodata=255,
    )
    s1_desc_aligned = align_to_reference(
        s1_desc, s1_desc_meta["transform"], s1_desc_meta["crs"], ref_meta,
        src_nodata=255, dst_nodata=255,
    )

    combined = (
        (s2_aligned == 1) | (s1_asc_aligned == 1) | (s1_desc_aligned == 1)
    ).astype(np.uint8)

    combined_meta = ref_meta.copy()
    combined_meta.update({"count": 1, "dtype": "uint8", "nodata": None})

    combined_folder = output_folder / "S1_S2" / str(year)
    combined_folder.mkdir(parents=True, exist_ok=True)
    combined_filename = combined_folder / f"s1_s2_asc_desc_combined_{year}_{thr}.tiff"

    with rasterio.open(combined_filename, "w", **combined_meta) as dest:
        dest.write(combined, 1)
    print("Wrote:", combined_filename)

    s2_bin = (s2_aligned == 1).astype(np.uint8)
    s1_bin = ((s1_asc_aligned == 1) | (s1_desc_aligned == 1)).astype(np.uint8)

    layer_folders = {
        "s2": output_folder / "S2" / str(year),
        "s1": output_folder / "S1" / str(year),
    }

    for name, arr in {"s2": s2_bin, "s1": s1_bin}.items():
        layer_folders[name].mkdir(parents=True, exist_ok=True)
        out_path = layer_folders[name] / f"{name}_aligned_{year}_{thr}.tiff"
        with rasterio.open(out_path, "w", **combined_meta) as dst:
            dst.write(arr, 1)
        print("Wrote:", out_path)


===== 2023 =====


/tmp/ipykernel_1861977/1856534901.py:17: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s1_asc = src.read(1)
/tmp/ipykernel_1861977/1856534901.py:20: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s1_desc = src.read(1)
/tmp/ipykernel_1861977/1856534901.py:23: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  s2 = src.read(1)


Wrote: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_asc_desc_combined_2023_0.4.tiff
Wrote: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/s2_aligned_2023_0.4.tiff
Wrote: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1/2023/s1_aligned_2023_0.4.tiff


### 6) Clip to glacier outlines from 1997

In [ ]:
mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"
glacier_mask = gpd.read_file(mask_filename)

def _write_tif(filename, array, crs, transform):
    out_meta = {
        "driver": "GTiff",
        "height": array.shape[0],
        "width": array.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
        "nodata": 0,
        "compress": "lzw",
    }
    with rasterio.open(filename, "w", **out_meta) as dst:
        dst.write(array.astype(np.uint8), 1)
    print(f"Saved: {filename}")

def process_binary_mask(binary_mask, transform, crs, glacier_mask,
                        disk_size=5, output_filename=None,
                        glacier_output_filename=None,
                        masked_input_filename=None):
    """Mask a binary raster with glacier polygons, apply morphological
    opening, and optionally save the result, the glacier mask, and the
    masked (pre-opening) input."""
    binary_mask = binary_mask.astype(np.uint8)

    meta = {
        "driver": "GTiff",
        "height": binary_mask.shape[0],
        "width": binary_mask.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
    }

    glacier_mask_reprojected = glacier_mask.to_crs(crs)

    with MemoryFile() as memfile:
        with memfile.open(**meta) as tmp:
            tmp.write(binary_mask, 1)

        with memfile.open() as src:
            masked_array, masked_transform = mask(
                src,
                glacier_mask_reprojected.geometry,
                crop=True,
                filled=False,
            )

    # True inside the glacier polygons
    glacier_area = ~np.ma.getmaskarray(masked_array[0])

    # save glacier mask 
    if glacier_output_filename is not None:
        _write_tif(glacier_output_filename, glacier_area, crs, masked_transform)

    # input clipped to glacier 
    binary = masked_array[0].filled(0).astype(np.uint8)
    if masked_input_filename is not None:
        _write_tif(masked_input_filename, binary, crs, masked_transform)

    struct = ski.morphology.disk(disk_size)
    opened = ski.morphology.opening(binary, struct).astype(np.uint8)
    opened[~glacier_area] = 0

    if output_filename is not None:
        _write_tif(output_filename, opened, crs, masked_transform)

    return opened, masked_transform, glacier_area

In [40]:
disk_sizes = [3,4,5]

for year in years:
    print(f"\n===== {year} =====")
    print(output_folder)

    inputs = {
        "s2": output_folder / "S2" / str(year) /
              f"s2_aligned_{year}_{thr}.tiff",

        "s1": output_folder / "S1" / str(year) /
              f"s1_aligned_{year}_{thr}.tiff",

        "combined": output_folder / "S1_S2" / str(year) /
                    f"s1_s2_asc_desc_combined_{year}_{thr}.tiff",
    }

    # Check that all inputs exist
    missing = [p.name for p in inputs.values() if not p.exists()]

    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    # Process each input raster
    for name, path in inputs.items():

        for i in disk_sizes:

            # Generate output filenames for this disk size
            outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"s2_glacier_{year}_{thr}_{i}.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"s1_glacier_{year}_{thr}_{i}.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"s1_s2_asc_desc_combined_glacier_{year}_{thr}_{i}.tiff",
            }

            glacier_outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"glacier_mask_s2_{thr}_{i}.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"glacier_mask_s1_{thr}_{i}.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"glacier_mask_s1_s2_{thr}_{i}.tiff",
            }

            # Read input raster
            with rasterio.open(path) as src:
                arr = src.read(1)
                transform = src.transform
                crs = src.crs

            # Process the raster
            process_binary_mask(
                arr,
                transform,
                crs,
                glacier_mask,
                disk_size=i,
                output_filename=outputs[name],
                glacier_output_filename=glacier_outputs[name]
            )

            print(f"Processed {name.upper()} for {year}, disk size {i}")


===== 2023 =====
/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results


/tmp/ipykernel_1861977/2424728616.py:55: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/glacier_mask_s2_0.4_3.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/s2_glacier_2023_0.4_3.tiff
Processed S2 for 2023, disk size 3


/tmp/ipykernel_1861977/2424728616.py:55: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/glacier_mask_s2_0.4_4.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/s2_glacier_2023_0.4_4.tiff
Processed S2 for 2023, disk size 4


...

Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/glacier_mask_s1_s2_0.4_3.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_asc_desc_combined_glacier_2023_0.4_3.tiff
Processed COMBINED for 2023, disk size 3


/tmp/ipykernel_1861977/2424728616.py:55: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/glacier_mask_s1_s2_0.4_4.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_asc_desc_combined_glacier_2023_0.4_4.tiff
Processed COMBINED for 2023, disk size 4


/tmp/ipykernel_1861977/2424728616.py:55: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/glacier_mask_s1_s2_0.4_5.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_asc_desc_combined_glacier_2023_0.4_5.tiff
Processed COMBINED for 2023, disk size 5


### Try out median filter

In [49]:
def apply_median_filter(
    binary_mask,
    transform,
    crs,
    meta,
    masked_input_filename,
    median_size,
    output_filename
):
    # Apply median filter
    median_arr = median_filter(
        binary_mask.astype(np.uint8),
        size=median_size
    )

    # Update metadata
    median_meta = meta.copy()
    median_meta.update({
        "height": median_arr.shape[0],
        "width": median_arr.shape[1],
        "transform": transform,
        "crs": crs,
        "count": 1,
        "dtype": "uint8",
    })

    # Reproject glacier polygons
    glacier_mask_reprojected = glacier_mask.to_crs(crs)

    # Clip median-filtered raster to glacier polygons
    with MemoryFile() as memfile:
        with memfile.open(**median_meta) as tmp:
            tmp.write(median_arr, 1)

        with memfile.open() as src:
            masked_array, masked_transform = mask(
                src,
                glacier_mask_reprojected.geometry,
                crop=True,
                filled=False,
            )

    # input clipped to glacier 
    binary = masked_array[0].filled(0).astype(np.uint8)

    if masked_input_filename is not None:
        _write_tif(masked_input_filename, binary, crs, masked_transform)

    print(f"Saved median-filtered raster: {output_filename}")

In [51]:
median_sizes = [3]

for year in years:

    print(f"\n===== {year} =====")

    # Input glacier masks
    inputs = {
        "s2": output_folder / "S2" / str(year) /
              f"s2_aligned_{year}_{thr}.tiff",

        "s1": output_folder / "S1" / str(year) /
              f"s1_aligned_{year}_{thr}.tiff",

        "combined": output_folder / "S1_S2" / str(year) /
                    f"s1_s2_asc_desc_combined_{year}_{thr}.tiff",
    }

    for name, input_file in inputs.items():

        if not input_file.exists():
            print(f"Missing input: {input_file}")
            continue

        for size in median_sizes:

            # Generate output filenames for this disk size
            outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"s2_glacier_{year}_{thr}_mean_{size}.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"s1_glacier_{year}_{thr}_mean_{size}.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"s1_s2_asc_desc_combined_glacier_{year}_{thr}_mean_{size}.tiff",
            }

            # Read input raster
            with rasterio.open(path) as src:
                arr = src.read(1)
                transform = src.transform
                crs = src.crs

            # Process the raster
            process_binary_mask(
                arr,
                transform,
                crs,
                glacier_mask,
                disk_size=i,
                output_filename=outputs[name],
                glacier_output_filename=glacier_outputs[name]
            )


===== 2023 =====


/tmp/ipykernel_1861977/482539562.py:41: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/glacier_mask_s2_0.4_5.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/s2_glacier_2023_0.4_mean_3.tiff


/tmp/ipykernel_1861977/482539562.py:41: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1/2023/glacier_mask_s1_0.4_5.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1/2023/s1_glacier_2023_0.4_mean_3.tiff


/tmp/ipykernel_1861977/482539562.py:41: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/glacier_mask_s1_s2_0.4_5.tiff
Saved: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_asc_desc_combined_glacier_2023_0.4_mean_3.tiff


### 8) Convert into polygons

In [62]:
def raster_to_polygons(binary_raster, transform, crs):
    """Convert pixels == 1 of a binary raster into polygons."""
    binary_raster = binary_raster.astype(np.uint8)
    mask = binary_raster == 1  

    geoms = [
        shape(geom)
        for geom, value in shapes(binary_raster, mask=mask, transform=transform)
    ]
    return gpd.GeoDataFrame({"value": [1] * len(geoms)}, geometry=geoms, crs=crs)


version = "v0.4_disk3"

layers = {
    # name: (input glacier raster, output shapefile)
    "s2": (
        lambda y: output_folder / "S2" / str(y) / f"s2_glacier_{y}_{thr}_3.tiff",
        lambda y: output_folder / "S2" / str(y) / f"s2_glacier_outlines_{y}_{version}.shp",
    ),
    "s1": (
        lambda y: output_folder / "S1" / str(y) / f"s1_glacier_{y}_{thr}_3.tiff",
        lambda y: output_folder / "S1" / str(y) / f"s1_glacier_outlines_{y}_{version}.shp",
    ),
    "combined": (
        #lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_asc_desc_combined_glacier_{y}_{thr}_{i}.tiff",
        #lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_glacier_outlines_{y}_{version}.shp", 
        lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_asc_desc_combined_glacier_{y}_{thr}_3.tiff",
        lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_glacier_outlines_{y}_{version}_3.shp",           
    ),
}

In [63]:
for year in years:
    print(f"\n===== {year} =====")

    in_paths = {name: build_in(year) for name, (build_in, _) in layers.items()}
    missing = [p.name for p in in_paths.values() if not p.exists()]
    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    for name, (_, build_out) in layers.items():
        with rasterio.open(in_paths[name]) as src:
            arr = src.read(1)
            transform = src.transform
            crs = src.crs

        gdf = raster_to_polygons(arr, transform, crs)

        out_path = build_out(year)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        gdf.to_file(out_path)

        print(f"{name}: {len(gdf)} polygons -> {out_path}")


===== 2023 =====


/tmp/ipykernel_1861977/105853942.py:12: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


s2: 228 polygons -> /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/2023/s2_glacier_outlines_2023_v0.4_disk3.shp


/tmp/ipykernel_1861977/105853942.py:12: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


s1: 421 polygons -> /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1/2023/s1_glacier_outlines_2023_v0.4_disk3.shp


/tmp/ipykernel_1861977/105853942.py:12: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


combined: 398 polygons -> /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S1_S2/2023/s1_s2_glacier_outlines_2023_v0.4_disk3_3.shp


### 9) Extract area in km2

In [64]:
rows = []

for year in years:
    for name, (_, build_out) in layers.items():
        path = build_out(year)
        if not path.exists():
            print(f"Missing: {path.name}, skipping.")
            continue

        gdf = gpd.read_file(path)

        if gdf.crs is None or not gdf.crs.is_projected:
            raise ValueError(f"{path.name}: CRS is not projected, area in m² would be wrong.")

        area_km2 = gdf.geometry.area.sum() / 1e6
        rows.append({
            "year": year,
            "layer": name,
            "n_polygons": len(gdf),
            "area_km2": round(area_km2, 3),
        })
        print(f"{year} {name:<9} {area_km2:8.2f} km²  ({len(gdf)} polygons)")

2023 s2           45.64 km²  (228 polygons)
2023 s1           60.71 km²  (421 polygons)
2023 combined     68.60 km²  (398 polygons)
